## Setup


In [ ]:
!pip install -q umap-learn


In [ ]:
import os, glob, json, time, urllib.request
from collections import defaultdict
from scipy.io import loadmat
from google.colab import drive
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.nn.functional as F
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score
import umap


In [ ]:
BASE = "/content"
DRIVE_PROJECT_DIR = f"{BASE}/drive/MyDrive/braid2"
INPUT_DIR = f"{DRIVE_PROJECT_DIR}/inputs"
OUTPUT_ROOT = f"{DRIVE_PROJECT_DIR}/outputs"
SUBJECT = "subj01"
BETA_DIR = f"{BASE}/subject01_visual_brain_responses"
CLIP_DIR = f"{BASE}/subject01_clip_image_embeddings"
EXP = f"{BASE}/nsd_expdesign.mat"
TRAINING_LOSS_FUNCTIONS = ["mse", "cos"]
EVAL_LOSS_FUNCTIONS = ["mse", "cos", "silhouette"]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCHS = 15
BATCH = 512
LR = 1e-4
DROPOUT = 0.5
TIMESTAMP = time.strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = f"{OUTPUT_ROOT}/{TIMESTAMP}"
SILHOUETTE_MAX = 1000
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:
drive.mount(f"{BASE}/drive")
!cp -r {INPUT_DIR}/subject01_visual_brain_responses /content/
!cp -r {INPUT_DIR}/subject01_clip_image_embeddings /content/
if not os.path.exists(EXP):
    urllib.request.urlretrieve("https://natural-scenes-dataset.s3.amazonaws.com/nsddata/experiments/nsd/nsd_expdesign.mat", EXP)

def session_id(path):
    return int(''.join(c for c in os.path.basename(path)[-6:] if c.isdigit()))

beta_sess = {session_id(p): p for p in glob.glob(f"{BETA_DIR}/{SUBJECT}_visualroi_session*.pt")}
clip_sess = {session_id(p): p for p in glob.glob(f"{CLIP_DIR}/{SUBJECT}_clip_embeds*.pt")}
sessions = sorted(set(beta_sess) & set(clip_sess))
betas = torch.cat([torch.load(beta_sess[s]).float() for s in sessions])
clips = torch.cat([torch.load(clip_sess[s]).float() for s in sessions])
mat = loadmat(EXP)
masterordering = mat["masterordering"].reshape(-1).astype(np.int64) - 1
subjectim = mat["subjectim"].astype(np.int64) - 1
imgbrick_ids = subjectim[int(SUBJECT[-2:]) - 1, masterordering]
shared_ids = set(mat["sharedix"].reshape(-1).astype(np.int64) - 1)
img_of = np.array([int(imgbrick_ids[(sessions[g // 750] - 1) * 750 + g % 750]) for g in range(len(betas))])
is_test = np.array([i in shared_ids for i in img_of])
rest_img = np.array(sorted(set(img_of[~is_test])))
np.random.RandomState(0).shuffle(rest_img)
val_img = set(rest_img[:int(0.05 * len(rest_img))])
in_val = np.array([i in val_img for i in img_of])
train_idx = np.where(~is_test & ~in_val)[0]
val_idx = np.where(~is_test & in_val)[0]
test_idx = np.where(is_test)[0]
beta_mean, beta_std = betas[train_idx].mean(0), betas[train_idx].std(0) + 1e-6
clip_mean, clip_std = clips[train_idx].mean(0), clips[train_idx].std(0) + 1e-6

def norm(b, c):
    return (b - beta_mean) / beta_std, (c - clip_mean) / clip_std

groups = defaultdict(list)
for g in test_idx:
    groups[int(img_of[g])].append(g)
test_img_ids = list(groups)
test_betas = torch.stack([betas[gs].mean(0) for gs in groups.values()])
test_clips = torch.stack([clips[gs[0]] for gs in groups.values()])
train_ds = TensorDataset(*norm(betas[train_idx], clips[train_idx]))
val_ds = TensorDataset(*norm(betas[val_idx], clips[val_idx]))
test_ds = TensorDataset(*norm(test_betas, test_clips))
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True)
train_eval_loader = DataLoader(train_ds, batch_size=BATCH)
val_loader = DataLoader(val_ds, batch_size=BATCH)
test_loader = DataLoader(test_ds, batch_size=BATCH)
print(f"device={DEVICE} output={OUTPUT_DIR}")
print(f"sessions={sessions} betas={tuple(betas.shape)} clips={tuple(clips.shape)}")
print(f"train={len(train_idx)} val={len(val_idx)} test_trials={len(test_idx)} test_images={len(test_img_ids)}")


## Model & Loss Functions


In [ ]:
class FMRIEncoderMLP(nn.Module):
    def __init__(self, input_dim, output_dim=1280, dropout=0.5):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(input_dim, 2048), nn.LayerNorm(2048), nn.ReLU(), nn.Dropout(dropout), nn.Linear(2048, 4096), nn.LayerNorm(4096), nn.ReLU(), nn.Dropout(dropout), nn.Linear(4096, output_dim))
    def forward(self, x):
        return self.net(x)

def loss_fn(name):
    if name == "mse":
        return nn.MSELoss()
    if name == "cos":
        return lambda pred, y: 1 - F.cosine_similarity(pred, y, dim=1).mean()
    raise ValueError(name)


## Training

In [ ]:
def collect(model, loader, raw=False):
    model.eval()
    P, T = [], []
    with torch.no_grad():
        for x, y in loader:
            pred = model(x.to(DEVICE))
            true = y.to(DEVICE)
            if raw:
                pred = pred * clip_std.to(DEVICE) + clip_mean.to(DEVICE)
                true = true * clip_std.to(DEVICE) + clip_mean.to(DEVICE)
            P.append(pred.cpu())
            T.append(true.cpu())
    return torch.cat(P), torch.cat(T)

def metric_value(name, pred, true):
    if name == "mse":
        return float(F.mse_loss(pred, true))
    if name == "cos":
        return float(1 - F.cosine_similarity(pred, true, dim=1).mean())
    if name == "silhouette":
        x = torch.cat([pred, true]).numpy()
        y = np.r_[np.zeros(len(pred)), np.ones(len(true))]
        if len(x) > SILHOUETTE_MAX:
            ids = np.random.RandomState(0).choice(len(x), SILHOUETTE_MAX, replace=False)
            x, y = x[ids], y[ids]
        return float(silhouette_score(x, y))
    raise ValueError(name)

def evaluate_metrics(model, loader):
    pred, true = collect(model, loader)
    return {name: metric_value(name, pred, true) for name in EVAL_LOSS_FUNCTIONS}

results = {}
for train_loss in TRAINING_LOSS_FUNCTIONS:
    run_dir = f"{OUTPUT_DIR}/mlp_{train_loss}"
    os.makedirs(run_dir, exist_ok=True)
    model = FMRIEncoderMLP(betas.shape[1], clips.shape[1], DROPOUT).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    objective = loss_fn(train_loss)
    history = {split: {metric: [] for metric in EVAL_LOSS_FUNCTIONS} for split in ["train", "val"]}
    print(f"\n=== train_loss={train_loss} ===")
    print("epoch " + " ".join([f"train_{m:>10s} val_{m:>10s}" for m in EVAL_LOSS_FUNCTIONS]))
    for epoch in range(1, EPOCHS + 1):
        model.train()
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            loss = objective(model(x), y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        train_scores = evaluate_metrics(model, train_eval_loader)
        val_scores = evaluate_metrics(model, val_loader)
        for metric in EVAL_LOSS_FUNCTIONS:
            history["train"][metric].append(train_scores[metric])
            history["val"][metric].append(val_scores[metric])
        print(f"{epoch:05d} " + " ".join([f"{train_scores[m]:12.4f} {val_scores[m]:12.4f}" for m in EVAL_LOSS_FUNCTIONS]))
    torch.save(model.state_dict(), f"{run_dir}/model.pth")
    with open(f"{run_dir}/history.json", "w") as f:
        json.dump(history, f, indent=2)
    for metric in EVAL_LOSS_FUNCTIONS:
        plt.figure(dpi=180)
        plt.plot(history["train"][metric], label="train")
        plt.plot(history["val"][metric], label="val")
        plt.xlabel("epoch")
        plt.ylabel(metric)
        plt.title(f"train loss={train_loss} eval={metric}")
        plt.grid(alpha=0.3)
        plt.legend()
        plt.savefig(f"{run_dir}/curve_{metric}.png", dpi=180, bbox_inches="tight")
        plt.show()
    results[train_loss] = {"model": model, "run_dir": run_dir, "history": history}


## Evaluation

In [ ]:
summary = {}
for train_loss, result in results.items():
    model = FMRIEncoderMLP(betas.shape[1], clips.shape[1], DROPOUT).to(DEVICE)
    model.load_state_dict(torch.load(f"{result['run_dir']}/model.pth", map_location=DEVICE))
    pred, true = collect(model, test_loader, raw=True)
    scores = {name: metric_value(name, pred, true) for name in EVAL_LOSS_FUNCTIONS}
    summary[train_loss] = scores
    print(f"\n=== eval_model={train_loss} ===")
    for name, value in scores.items():
        print(f"{name:>12s}: {value:.4f}")
    pred_np, true_np = pred.numpy(), true.numpy()
    emb = np.concatenate([pred_np, true_np])
    colors = ["green"] * len(pred_np) + ["blue"] * len(true_np)
    fig, axes = plt.subplots(1, 3, figsize=(18, 5), dpi=150)
    pca = PCA(n_components=2).fit_transform(emb)
    tsne = TSNE(n_components=2, init="pca", perplexity=30, random_state=0).fit_transform(emb)
    umap_xy = umap.UMAP(n_components=2, random_state=0).fit_transform(emb)
    for ax, xy, title in zip(axes, [pca, tsne, umap_xy], ["PCA", "t-SNE", "UMAP"]):
        ax.scatter(xy[:, 0], xy[:, 1], c=colors, s=6, alpha=0.5)
        ax.set_title(title)
        ax.set_xticks([])
        ax.set_yticks([])
    axes[-1].legend([plt.Line2D([], [], marker="o", ls="", color="green"), plt.Line2D([], [], marker="o", ls="", color="blue")], ["fMRI-predicted", "CLIP image"])
    fig.suptitle(f"test manifold train loss={train_loss}")
    plt.tight_layout()
    plt.savefig(f"{result['run_dir']}/manifold_umap_pca_tsne.png", dpi=150, bbox_inches="tight")
    plt.show()
    K = 50
    V = PCA(n_components=K).fit(clips[train_idx].numpy()).components_
    def cum_var(x):
        xc = x - x.mean(0)
        return np.cumsum(((xc @ V.T) ** 2).sum(0) / len(xc)) / ((xc ** 2).sum() / len(xc))
    ks = np.arange(1, K + 1)
    plt.figure(figsize=(9, 5), dpi=180)
    plt.plot(ks, cum_var(clips[train_idx].numpy()), "o-", ms=3, label="train true CLIP")
    plt.plot(ks, cum_var(test_clips.numpy()), "s-", ms=3, label="test true CLIP")
    plt.plot(ks, cum_var(pred_np), "s-", ms=3, label="fMRI-predicted")
    plt.xlabel("principal components")
    plt.ylabel("cumulative variance")
    plt.ylim(0, 1)
    plt.grid(alpha=0.3, ls="--")
    plt.legend()
    plt.title(f"variance train loss={train_loss}")
    plt.savefig(f"{result['run_dir']}/scree_plot.png", dpi=180, bbox_inches="tight")
    plt.show()
with open(f"{OUTPUT_DIR}/summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(f"\nsummary saved to {OUTPUT_DIR}/summary.json")
